In [ ]:
import numpy as np
from abc import ABC, abstractmethod
# Inicializamos la clase abstracta Layer, que tiene los métodos forward y backward que se implementarán en classes Lineal y Relu
class Layer(ABC):
    

    @abstractmethod
    def forward(batch_input):
        pass

    @abstractmethod
    def backward(error):
        pass


In [ ]:
#Inicialitzamos la clase Lineal y Relu, que hereda de Layer. Esta clase implementará los métodos forward y backward

class Lineal(Layer):
    def __init__(self, input_size, batch_size):
        self.w = np.random.randn(input_size, batch_size) * np.sqrt(2 / input_size)
        self.b = np.zeros((1, batch_size))           

    def forward(self, x):
        self.input = x 
        return x @ self.w + self.b

    #Este metodo calcula la pèrdida restando la prediccion de la true, y devuelve la media de los errores al cuadrado
    def loss(spasself, y_pred, y_true):

        return np.mean((y_pred - y_true) ** 2)
    
    def backward(self, error):
        # Multiplicamos el error por el input de la anterior capa para obtener el gradiente de los pesos
        self.grad_w = self.input.T @ error
        
        self.grad_b = error.sum(axis=0, keepdims=True)

        # Devolvemos el error multiplicado por el peso de la capa para que la anterior capa pueda calcular su gradiente
        return error @ self.w.T
    
    def set(self, lr):
        self.w -= lr * self.grad_w
        self.b -= lr * self.grad_b

class Relu(Layer):
    def __init__(self, input_size, batch_size):
            self.w = np.random.randn(input_size, batch_size) * np.sqrt(2 / input_size)
            self.b = np.zeros((1, batch_size))           
            
    def forward(self, x):
        self.input = x                  
        self.z = x @ self.w + self.b    
        return np.maximum(0, self.z)
    
    def backward(self, error):
        error = error * (self.z > 0)        
        self.grad_w = self.input.T @ error
        self.grad_b = error.sum(axis=0, keepdims=True)
        return error @ self.w.T

    def set(self, lr):
        self.w -= lr * self.grad_w
        self.b -= lr * self.grad_b

In [ ]:
import idx2numpy
import numpy as np

imgs = idx2numpy.convert_from_file('data/Original dataset/train-images.idx3-ubyte')
labels = idx2numpy.convert_from_file('data/Original dataset/train-labels.idx1-ubyte')

print(imgs.shape)

# Aplanar imagen para poder multiplicar con matriz de pesos y normalizar los valores a 0 i 1
X = imgs.reshape(len(imgs), -1).astype(np.float64) / 255.0
print(X.shape)

# Adaptamos también para poder comparar el loss con la predicción 
Y = np.eye(10)[labels]

print(Y.shape)

# Inicializamos las hidden layers y la capa de salida
h1 = Lineal(784, 128)
h2 = Relu(128, 64)
out = Lineal(64, 10)



batch = 64
epochs = 10
lr = 0.1

for epoch in range(epochs):
    idx = np.random.permutation(len(X))
    losses = []

    for i in range(0, len(X), batch):
        b = idx[i:i + batch]
        x_batch, y_batch = X[b], Y[b]

        # forward
        h1_out = h1.forward(x_batch)
        h2_out = h2.forward(h1_out)
        out_out = out.forward(h2_out)
        losses.append(out.loss(out_out, y_batch))

        # backward
        
        grad = 2 * (out_out - y_batch) / out_out.size
        grad = out.backward(grad)
        grad = h2.backward(grad)
        grad = h1.backward(grad)

        # actualizar pesos
        out.set(lr)
        h2.set(lr)
        h1.set(lr)

    pred = out.forward(h2.forward(h1.forward(X)))
    acc = np.mean(np.argmax(pred, axis=1) == labels)
    print(f"Época {epoch + 1}/{epochs}: loss={np.mean(losses):.4f}  acc={acc:.4f}")


(60000, 28, 28)
(60000, 784)
(60000, 10)
std w h1: 0.050531354562196075
b h1 todo cero: True
Loss inicial: 0.4613644763281709
Época 1/10: loss=0.0470  acc=0.8784
Época 2/10: loss=0.0272  acc=0.9036
Época 3/10: loss=0.0226  acc=0.9139
Época 4/10: loss=0.0204  acc=0.9193
Época 5/10: loss=0.0191  acc=0.9244
Época 6/10: loss=0.0182  acc=0.9276
Época 7/10: loss=0.0175  acc=0.9301
Época 8/10: loss=0.0169  acc=0.9326
Época 9/10: loss=0.0164  acc=0.9345
Época 10/10: loss=0.0159  acc=0.9368
